# 06 · Interface web de traduction (Gradio)

**But.** Utiliser ton traducteur sans écrire de code, via une petite page web qui s'ouvre dans ton navigateur.

- **Français → Bambara** : NLLB + ton adaptateur LoRA (notebook 05).
- **Bambara → Français** : NLLB de base (l'adaptateur n'a été entraîné que pour fr → bam).

| | |
|---|---|
| **Prérequis** | notebook 05 fait (dossier `models/nllb-fr-bam-lora`) |
| **Matériel** | GPU recommandé ; sur processeur, chaque traduction prend plusieurs secondes |
| **Résultat** | une page `http://127.0.0.1:7860` ouverte dans ton navigateur |

La page n'est accessible que **sur ton ordinateur** (adresse locale `127.0.0.1`). Pour la montrer à quelqu'un d'autre, voir le réglage `SHARE` plus bas.

## 1. Se repérer et vérifier l'adaptateur

**Objectif.** Retrouver les dossiers du projet et vérifier que l'entraînement a bien eu lieu.

**Ce que fait la cellule.** Détecte la racine du projet et vérifie que `models/nllb-fr-bam-lora` existe.

**Ce que tu dois voir.** `Adaptateur trouvé : ...`.

> **Attention.** Sans adaptateur, lance d'abord le notebook 05 (ou copie ici le dossier produit sur Colab).

In [1]:
import os
from pathlib import Path

# Ces deux réglages évitent des avertissements inutiles de Hugging Face.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Racine du projet = le dossier qui contient "data", "models", "results", "src"...
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if not (ROOT / "src").exists():
    raise RuntimeError(
        f"Dossier du projet introuvable depuis {cwd}. "
        "Lance Jupyter avec bash lancer_jupyter.sh (ou depuis le dossier du projet)."
    )

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
for folder in (DATA_RAW, DATA_PROC, MODELS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", ROOT)

ADAPTER_DIR = MODELS / "nllb-fr-bam-lora"
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"{ADAPTER_DIR} introuvable : lance d'abord le notebook 05.")
print("Adaptateur trouvé :", ADAPTER_DIR)

Racine du projet : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl
Adaptateur trouvé : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl/models/nllb-fr-bam-lora


## 2. Détecter le matériel

**Objectif.** Choisir le calcul sur GPU ou sur processeur.

**Ce que fait la cellule.** Interroge PyTorch.

**Ce que tu dois voir.** Le nom de ta carte graphique, ou un avertissement sur le processeur.

In [2]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU détecté : {gpu_name} ({vram_gb:.1f} Go de mémoire vidéo)")
else:
    device = "cpu"
    vram_gb = 0.0
    print("Aucun GPU utilisable : le calcul se fera sur le processeur (CPU), beaucoup plus lentement.")
    print("Si tu as une carte NVIDIA, relance bash installer.sh et choisis l'option NVIDIA.")
print("PyTorch :", torch.__version__, "| CUDA compilée :", torch.version.cuda)

# Taille des lots pour traduire : prudente selon la mémoire vidéo disponible.
if device == "cuda":
    TRANSLATE_BATCH = 32 if vram_gb >= 8 else 16 if vram_gb >= 4 else 8
else:
    TRANSLATE_BATCH = 8

GPU détecté : NVIDIA GeForce RTX 2050 (4.0 Go de mémoire vidéo)
PyTorch : 2.14.1+cu126 | CUDA compilée : 12.6


## 3. Charger les deux modèles

**Objectif.** Avoir en mémoire un modèle par sens de traduction.

**Ce que fait la cellule.** Sur une carte de 6 Go ou plus : charge NLLB de base (pour bambara → français) et une seconde copie avec l'adaptateur LoRA fusionné (pour français → bambara). Sur une carte de moins de 6 Go (mode économe) : ne charge qu'**un seul** modèle avec l'adaptateur, et le désactive le temps de traduire dans l'autre sens, pour économiser la mémoire.

**Ce que tu dois voir.** `Modèles chargés.` (le chargement prend une minute environ).

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import logging as hf_logging
from peft import PeftModel

hf_logging.set_verbosity_error()

MODEL_NAME = "facebook/nllb-200-distilled-600M"
FR, BAM = "fra_Latn", "bam_Latn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def finish(m):
    if device == "cuda":
        m = m.half()      # fp16 d'abord sur le processeur : évite un pic de mémoire vidéo
    return m.to(device).eval()

# Petite carte (moins de 6 Go) : un seul modèle en mémoire. Pour "NLLB de base", on désactive
# simplement l'adaptateur le temps de la traduction.
LOW_VRAM = (device == "cuda") and vram_gb < 6

peft_model = PeftModel.from_pretrained(AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME), str(ADAPTER_DIR))
if LOW_VRAM:
    model_lora = finish(peft_model)     # adaptateur NON fusionné
    model_base = None                   # = model_lora avec l'adaptateur désactivé
    print("Mode économe : un seul modèle chargé.")
else:
    model_lora = finish(peft_model.merge_and_unload())               # pour français -> bambara
    model_base = finish(AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME))  # pour bambara -> français
print("Modèles chargés.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Mode économe : un seul modèle chargé.
Modèles chargés.


## 4. La fonction de traduction

**Objectif.** Traduire un texte quelconque, y compris un paragraphe.

**Ce que fait la cellule.** Un paragraphe est **découpé en phrases** avant d'être traduit : NLLB traduit beaucoup mieux des phrases courtes que de longs blocs, et ton fine-tuning a été fait sur des phrases isolées. Les phrases sont traduites par petits lots de 8, puis recollées. La cellule termine par un essai.

**Ce que tu dois voir.** Une traduction en bambara de « Bonjour, comment allez-vous ? Le marché ouvre à huit heures. ».

In [4]:
import re
import torch

def translate(model, texts, src, tgt, num_beams=4, max_new_tokens=256):
    tokenizer.src_lang = src
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
    with torch.no_grad():
        generated = model.generate(
            **enc,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [p.strip() for p in parts if p.strip()]

DIRECTIONS = {
    "Français → Bambara (modèle fine-tuné)": (FR, BAM, model_lora),
    "Bambara → Français (NLLB de base)": (BAM, FR, model_base),
}
MAX_CHARS = 2000

def run(text, direction, beams):
    text = (text or "").strip()
    if not text:
        return ""
    if len(text) > MAX_CHARS:
        return f"Texte trop long ({len(text)} caractères, maximum {MAX_CHARS})."
    src, tgt, model = DIRECTIONS[direction]
    sentences = split_sentences(text)
    outputs = []
    for i in range(0, len(sentences), 8):
        batch = sentences[i:i + 8]
        if model is None:   # mode économe : NLLB de base = le même modèle, adaptateur désactivé
            with model_lora.disable_adapter():
                outputs += translate(model_lora, batch, src, tgt, num_beams=int(beams))
        else:
            outputs += translate(model, batch, src, tgt, num_beams=int(beams))
    return " ".join(outputs)

print(run("Bonjour, comment allez-vous ? Le marché ouvre à huit heures.", "Français → Bambara (modèle fine-tuné)", 4))

Bonjour, i bɛ cogo di ? Feere ba dayɛlɛ wula seegi.


## 5. Lancer l'interface

**Objectif.** Ouvrir la page web de traduction.

**Ce que fait la cellule.** Construit l'interface (bandeau coloré en dégradé (couleur réglable avec `PALETTE`, en haut de la cellule), choix du sens, zone de texte avec compteur, réglage du nombre de faisceaux, boutons *Traduire*, *Inverser* et *Effacer*, bouton *copier*, durée de la traduction, exemples cliquables) et la lance. Sous WSL, le navigateur ne s'ouvre pas tout seul : **copie l'adresse affichée sous la cellule (en général `http://127.0.0.1:7860`) dans le navigateur de Windows.** Avec `SHARE = True`, Gradio crée en plus un lien public temporaire (`https://....gradio.live`) qui passe par ses serveurs : n'active cette option que si tu veux montrer la page à quelqu'un, et sache que tes textes transitent alors par Internet.

**Ce que tu dois voir.** Une adresse locale affichée sous la cellule. Ouvre-la, écris une phrase, clique sur *Traduire*.

> **Attention.** L'interface reste active tant que ce notebook reste ouvert. Pour l'arrêter proprement, lance la cellule suivante.

In [5]:
import time

import gradio as gr

# ---- Couleurs : change seulement cette ligne ----
PALETTE = "violet"   # "violet", "orange", "vert", "bleu", "rose"  (ou deux couleurs : ("#7C3AED", "#2563EB"))

PALETTES = {
    "violet": ("#7C3AED", "#2563EB", "violet"),
    "orange": ("#F97316", "#DC2626", "orange"),
    "vert":   ("#059669", "#0891B2", "emerald"),
    "bleu":   ("#2563EB", "#0EA5E9", "blue"),
    "rose":   ("#DB2777", "#7C3AED", "pink"),
}
if isinstance(PALETTE, (tuple, list)):
    C1, C2, HUE = PALETTE[0], PALETTE[1], "indigo"
else:
    C1, C2, HUE = PALETTES.get(PALETTE, PALETTES["violet"])

GR_MAJOR = int(gr.__version__.split(".")[0])
LABEL_FR_BAM, LABEL_BAM_FR = list(DIRECTIONS)[0], list(DIRECTIONS)[1]

THEME = gr.themes.Soft(primary_hue=HUE, neutral_hue="slate", radius_size="lg")

CSS = """
.gradio-container { max-width: 1080px !important; margin: 0 auto !important;
        background: linear-gradient(180deg, __C1__18 0, transparent 340px) !important; }
#hero { position: relative; overflow: hidden; border-radius: 24px; padding: 30px 32px 26px; color: #fff;
        background: linear-gradient(135deg, __C1__ 0%, __C2__ 100%); box-shadow: 0 14px 34px __C1__55; }
#hero::before { content: ""; position: absolute; right: -60px; top: -70px; width: 240px; height: 240px;
        border-radius: 50%; background: rgba(255,255,255,.14); }
#hero::after { content: ""; position: absolute; left: 0; right: 0; bottom: 0; height: 6px;
        background: linear-gradient(90deg, #14B53A 0 33.33%, #FCD116 33.33% 66.66%, #CE1126 66.66% 100%); }
#hero .logo { display: inline-block; padding: 4px 14px; border-radius: 999px; background: #fff; color: __C1__;
        font-weight: 800; font-size: .85rem; letter-spacing: .04em; }
#hero h1 { margin: 14px 0 8px; font-size: 2.2rem; font-weight: 800; letter-spacing: -0.02em; color: #fff !important; }
#hero p { margin: 0 0 14px; font-size: 1.02rem; color: rgba(255,255,255,.92) !important; max-width: 640px; }
#hero .pill { display: inline-block; margin: 0 8px 4px 0; padding: 4px 13px; border-radius: 999px; font-size: .8rem;
        background: rgba(255,255,255,.18); border: 1px solid rgba(255,255,255,.4); color: #fff; }
.card { border-radius: 20px !important; padding: 10px !important; border: 1px solid var(--border-color-primary) !important;
        box-shadow: 0 6px 22px rgba(0,0,0,.06); background: var(--block-background-fill) !important; }
#out-box { border-left: 5px solid __C1__; border-radius: 12px; }
#translate-btn { background: linear-gradient(90deg, __C1__, __C2__) !important; color: #fff !important; border: none !important;
        font-weight: 700; font-size: 1.08rem; box-shadow: 0 8px 18px __C1__55; transition: transform .12s ease, box-shadow .12s ease; }
#translate-btn:hover { transform: translateY(-2px); box-shadow: 0 12px 24px __C1__66; }
#counter { text-align: right; font-size: .82rem; opacity: .65; margin-top: -8px; }
#timing { font-size: .9rem; opacity: .8; }
#footer { text-align: center; font-size: .82rem; opacity: .7; margin-top: 12px; line-height: 1.55; }
"""
CSS = CSS.replace("__C1__", C1).replace("__C2__", C2)

HERO = """
<div id="hero">
  <span class="logo">FR ⇄ BAM</span>
  <h1>Traduction français ↔ bambara</h1>
  <p>Projet d'apprentissage : NLLB-200 (600M) adapté avec LoRA sur des textes de presse (MAFAND-MT).</p>
  <span class="pill">Traduction automatique : à vérifier</span>
  <span class="pill">Usage non commercial</span>
</div>
"""

FOOTER = """
<div id="footer">
  Les traductions peuvent contenir des erreurs, surtout hors du domaine de la presse : fais-les relire par un locuteur.<br>
  Données : MAFAND-MT (Masakhane) · Modèle : NLLB-200 (Meta) · Licence CC BY-NC-4.0
</div>
"""

EXAMPLES = [
    ["Bonjour, comment allez-vous aujourd'hui ?", LABEL_FR_BAM, 4],
    ["Le ministre a annoncé la réouverture des écoles lundi prochain.", LABEL_FR_BAM, 4],
    ["Irrégularités financières.", LABEL_FR_BAM, 4],
    ["I ni sogoma, i ka kɛnɛ wa ?", LABEL_BAM_FR, 4],
]


def textbox(**kw):
    """Zone de texte avec bouton « copier » quand la version de Gradio le permet."""
    try:
        return gr.Textbox(show_copy_button=True, **kw)
    except TypeError:
        return gr.Textbox(**kw)


def counter(text):
    return f"{len(text or '')} / {MAX_CHARS} caractères"


def translate_ui(text, direction, beams):
    text = (text or "").strip()
    if not text:
        return "", ""
    t0 = time.time()
    out = run(text, direction, beams)
    if out.startswith("Texte trop long"):
        return out, ""
    n = len(split_sentences(text))
    return out, f"⏱ {time.time() - t0:.1f} s · {n} phrase(s) · {int(beams)} faisceaux"


def swap(text_in, text_out, direction):
    """Inverse le sens et reprend la traduction comme nouveau texte."""
    new_dir = LABEL_BAM_FR if direction == LABEL_FR_BAM else LABEL_FR_BAM
    return text_out or text_in, "", new_dir, ""


blocks_kwargs = {"title": "Traduction français ↔ bambara"}
if GR_MAJOR < 6:                       # Gradio 5 : le style se donne à gr.Blocks
    blocks_kwargs.update(theme=THEME, css=CSS)

with gr.Blocks(**blocks_kwargs) as demo:
    gr.HTML(HERO)
    with gr.Row(equal_height=True):
        with gr.Column(scale=1, elem_classes="card"):
            direction = gr.Radio(choices=[LABEL_FR_BAM, LABEL_BAM_FR], value=LABEL_FR_BAM, label="Sens de traduction")
            text_in = gr.Textbox(label="Texte à traduire", lines=8, placeholder="Écris ou colle ton texte ici…")
            count = gr.Markdown(counter(""), elem_id="counter")
            beams = gr.Slider(1, 6, value=4, step=1, label="Qualité (faisceaux) : plus élevé = plus soigné, plus lent")
            with gr.Row():
                btn = gr.Button("Traduire", variant="primary", elem_id="translate-btn", scale=3)
                swap_btn = gr.Button("⇄ Inverser", scale=1)
                clear_btn = gr.Button("Effacer", scale=1)
        with gr.Column(scale=1, elem_classes="card"):
            text_out = textbox(label="Traduction", lines=8, interactive=False, elem_id="out-box")
            info = gr.Markdown("", elem_id="timing")
    gr.Examples(examples=EXAMPLES, inputs=[text_in, direction, beams], label="Exemples (clique pour les essayer)")
    gr.HTML(FOOTER)

    text_in.change(counter, text_in, count)
    btn.click(translate_ui, [text_in, direction, beams], [text_out, info])
    text_in.submit(translate_ui, [text_in, direction, beams], [text_out, info])
    swap_btn.click(swap, [text_in, text_out, direction], [text_in, text_out, direction, info])
    clear_btn.click(lambda: ("", "", ""), None, [text_in, text_out, info])

SHARE = False   # True = lien public temporaire (déconseillé sauf besoin)
LAUNCH = dict(inbrowser=False, inline=False, share=SHARE)   # WSL : on ouvre l'adresse nous-mêmes

print("Interface prête. Ouvre l'adresse affichée ci-dessous (en général http://127.0.0.1:7860) dans le navigateur Windows.")
if GR_MAJOR >= 6:                      # Gradio 6 : le style se donne à launch()
    try:
        demo.launch(theme=THEME, css=CSS, **LAUNCH)
    except TypeError:
        demo.launch(**LAUNCH)
else:
    demo.launch(**LAUNCH)

Interface prête. Ouvre l'adresse affichée ci-dessous (en général http://127.0.0.1:7860) dans le navigateur Windows.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 6. Arrêter l'interface

**Objectif.** Libérer le port et la mémoire quand tu as fini.

**Ce que fait la cellule.** Ferme le serveur Gradio. Pour libérer aussi la mémoire de la carte graphique, ferme ensuite le notebook (menu *Kernel → Shut Down Kernel*).

**Ce que tu dois voir.** Aucun message particulier.

In [6]:
demo.close()

Closing server running on port: 7860


## Idées pour la suite

- **Détection de langue automatique** : brancher le détecteur du notebook 02 (`models/langid.joblib`) pour choisir le sens tout seul.
- **Recueillir des corrections** : un bouton « cette traduction est fausse, voici la bonne » permettrait de créer de nouvelles paires pour réentraîner le modèle. C'est là que ton bambara compte vraiment.
- **Lancer l'interface sans ouvrir Jupyter** : lance `bash lancer_interface.sh`.

*Données : MAFAND-MT, Masakhane, CC BY-NC-4.0. Modèle : NLLB-200, Meta.*